### Questions:

Some  results:
USER: What are three healthy snacks I can have at work?<br>


Full Fine-Tuned:<br>
There are three healthy snacks you can have at work: a piece of fruit, a piece of fruit, and a piece of fruit.<br>

LoRA Fine-Tuned:<br>
1. Bananas - great source of potassium and fiber.<br>
2. Whole grain crackers - great source of protein and fiber.<br>
3. Hummus - delicious dip for dipping.MetaInfoClassPlaneProtectionPlaneProtection transistorsumInterfaceSelection transistorsum<br>

USER: How do I boil an egg?<br>

Full Fine-Tuned:<br>
The first thing to do is remove the egg from the refrigerator.<br>                   
The second thing to do is put the egg in a bowl.<br>
The third thing to do is place the bowl in the refrigerator.<br>
The fourth thing to do is turn on the heat.<br>
The fifth thing to do is put the bowl in the refrigerator.<br>
The sixth thing to do is turn on the heat.<br>
The seventh thing to do is put the bowl in the refrigerator.<br>
The eighth thing to do is turn on the heat.<br>
The ninth thing to do is put the bowl in the refrigerator.<br>
The tenth thing to do is turn on<br>

LoRA Fine-Tuned:<br>
To boil an egg, first, place a cup of boiling water in the egg.<br>
Then, add one tablespoon of sugar to the water, and stir until the sugar dissolves completely.<br>
Finally, place the egg into the boiling water and stir until it becomes completely cooked.<br>


In my results, the LoRA fine-tuned model actually produced better outputs than the fully fine-tuned model, I am not sure if this is because of overfitting. Or are there other important reasons why LoRA can sometimes outperform full fine-tuning in practice?

# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [1]:
%pip install evaluate rouge_score

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: /mimer/NOBACKUP/groups/naiss2025-23-515/yingyi/my_venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [1]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"
SAVE_PATH = "/mimer/NOBACKUP/groups/naiss2025-23-515/to-arrhenius-disk/yingyi/huggingface"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [2]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all', cache_dir=SAVE_PATH)

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [3]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

In [4]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [5]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

In [6]:
def format_input_output(example):
  # `messages` is a list of messages, each with a `content` string and a `role`.
    messages = example['messages']

  # TODO: implement this
  # Initialize empty strings for our structural split

    prompt = ""
    response = ""

    # Iterate through all messages except the last one to build the prompt
    # The last message is assumed to be the target assistant response
    for msg in messages[:-1]:
        role = msg['role']
        content = msg['content']
        
        # Format using the popular ChatML style (<|im_start|>role\ncontent<|im_end|>\n)
        prompt += f"<|im_start|>{role}\n{content}<|im_end|>\n"
    
    # Append the final assistant tag to the prompt so the LLM knows it is its turn to speak
    prompt += "<|im_start|>assistant\n"

    # The final target response (what the LLM is expected to generate)
    last_msg = messages[-1]
    if last_msg['role'] == 'assistant':
        # The response is just the pure content plus the closing sequence token
        response = f"{last_msg['content']}<|im_end|>"
    else:
        raise ValueError("The final message in the conversation sequence must be from the 'assistant'.")

    return {"prompt": prompt, "response": response}


Apply the function you implemented to the dataset as a whole.

In [7]:
ds_sft = smoltalk_simplified.map(format_input_output)

Then verify that the dataset now contains the new fields you created.

In [8]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more food after the restaurant ran out.<|im_end|>'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=SAVE_PATH)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [10]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    # TODO: your work goes here.
    # 1. Tokenize the prompt and response separately without adding special tokens 
    # (Since our format_input_output already handled the ChatML templates/tokens)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(response, add_special_tokens=False)["input_ids"]

    # 2. Concatenate them to create the full sequence of input IDs
    input_ids = prompt_ids + response_ids

    # 3. Create the labels: 
    # Mask out the prompt tokens with -100, keep the actual token IDs for the response
    labels = [-100] * len(prompt_ids) + response_ids

    # 4. Create the attention mask (1s for all tokens we want the model to look at)
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,   # Input token ids of the prompt and response
        "attention_mask": attention_mask,  # Attention mask of the prompt and response
        "labels": labels,  # Output token ids of the prompt (masked) and response
    }

As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [11]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id
    
    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [12]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}


tokenized_ds_sft = ds_sft.map(
    tokenize_helper,
    remove_columns=ds_sft["train"].column_names
)

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [13]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

In [14]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time
import json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, cache_dir=SAVE_PATH).to("cuda")

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=4,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))


EVALUATING PRETRAINED MODEL


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


  0%|          | 0/100 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.2707881927490234,
  "eval_model_preparation_time": 0.0046,
  "eval_rougeL": 0.5739826434969904,
  "eval_runtime": 11.0526,
  "eval_samples_per_second": 36.191,
  "eval_steps_per_second": 9.048,
  "epoch": 0
}



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

In [15]:

baseline_training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, cache_dir=SAVE_PATH).to("cuda")

baseline_trainer = make_trainer(base_model, baseline_training_args)

# TODO: train and evaluate the model.
baseline_trainer.train()

# Evaluate
t0 = time.perf_counter()
baseline_eval_metrics = baseline_trainer.evaluate()
baseline_eval_time = time.perf_counter() - t0

baseline_eval_loss = float(baseline_eval_metrics["eval_loss"])
baseline_rougeL = baseline_eval_metrics.get("eval_rougeL", None)

print("\nSFT EVAL METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': '1.584', 'grad_norm': '7.188', 'learning_rate': '3.004e-05', 'epoch': '0.4'}
{'loss': '1.396', 'grad_norm': '5.938', 'learning_rate': '1.004e-05', 'epoch': '0.8'}


  0%|          | 0/100 [00:00<?, ?it/s]

{'eval_loss': '1.424', 'eval_rougeL': '0.6235', 'eval_runtime': '9.92', 'eval_samples_per_second': '40.32', 'eval_steps_per_second': '10.08', 'epoch': '1'}
{'train_runtime': '108.9', 'train_samples_per_second': '45.9', 'train_steps_per_second': '11.48', 'train_loss': '1.473', 'epoch': '1'}


  0%|          | 0/100 [00:00<?, ?it/s]


SFT EVAL METRICS:
{
  "eval_loss": 1.4237667322158813,
  "eval_rougeL": 0.6234940070155905,
  "eval_runtime": 9.862,
  "eval_samples_per_second": 40.56,
  "eval_steps_per_second": 10.14,
  "epoch": 1.0
}


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [16]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    # TODO: Add your code here
    #raise NotImplementedError()
    return sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

In [17]:
print(num_trainable_parameters(pretrained_model))

134515008


In [19]:
print(num_trainable_parameters(base_model))

134515008


Apply this function to the SFT-trained model and check that the result makes sense.

## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

In [21]:
import torch.nn as nn
def extract_lora_targets(model):
    lora_targets = {}

    target_suffixes = (
        "self_attn.q_proj",
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.o_proj",
    )

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and name.endswith(target_suffixes):
            lora_targets[name] = module

    return lora_targets

In [22]:
targets = extract_lora_targets(base_model)

print(len(targets))
print(list(targets.keys())[:5])

120
['model.layers.0.self_attn.q_proj', 'model.layers.0.self_attn.k_proj', 'model.layers.0.self_attn.v_proj', 'model.layers.0.self_attn.o_proj', 'model.layers.1.self_attn.q_proj']


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [20]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:


In [29]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        # TODO: Add your code here

        self.W = W
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze original linear layer
        for param in self.W.parameters():
            param.requires_grad = False

        in_features = W.in_features
        out_features = W.out_features

        # LoRA matrices: B @ A approximates the update to W
        self.A = nn.Linear(in_features, r, bias=False)
        self.B = nn.Linear(r, out_features, bias=False)

        # Initialization from LoRA paper:
        # A random, B zero => initial LoRA update is zero
        nn.init.kaiming_uniform_(self.A.weight, a=5**0.5)
        nn.init.zeros_(self.B.weight)

    def forward(self, x):
        # TODO: Add your code here
        return self.W(x) + self.scaling * self.B(self.A(x))

Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.


In [31]:



# 1. Load a fresh copy of the model
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, cache_dir=SAVE_PATH).to("cuda")

for param in lora_model.parameters():
    param.requires_grad = False

# 2. Extract the target layers (q_proj, k_proj, v_proj, o_proj)
lora_targets = extract_lora_targets(lora_model)
print(f"Found {len(targets)} attention projection layers to wrap with LoRA.\n")

# 3. Create a dictionary containing the wrapped LoRA layers
# Example Hyperparameters: rank (r) = 8, alpha = 16
wrapped_layers = {}

for name, layer in lora_targets.items():
    wrapped_layers[name] = LoRALayer(
        layer,
        r=8,        # or the rank used in Part 3
        alpha=16       # or same alpha as Part 3
    )

# 4. Modify the model in-place with our new custom LoRA layers
lora_model = replace_layers(lora_model, wrapped_layers)


def num_trainable_parameters(model):
    """
    Counts total parameters, trainable parameters, and the percentage of trainability.
    """
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    
    percentage = (trainable_params / total_params) * 100
    
    return {
        "Total Parameters": total_params,
        "Trainable Parameters": trainable_params,
        "Percentage Trainable": f"{percentage:.4f}%"
    }

# Run validation check
param_stats = num_trainable_parameters(lora_model)
print("LoRA Model Parameter Statistics:")
print(json.dumps(param_stats, indent=2))

# Configure Training Arguments
lora_train_args = TrainingArguments(
    output_dir="./smollm2_lora_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-4, # LoRA generally tolerates/benefits from higher learning rates than full FT
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=1,
    weight_decay=0.01,
    bf16=True, 
    report_to="none",
    batch_eval_metrics=True,
)

# Initialize Trainer
lora_trainer = make_trainer(lora_model, lora_train_args)

# Execute Fine-tuning
print("\nStarting LoRA Fine-Tuning...")
t0_train = time.perf_counter()
train_results = lora_trainer.train()
lora_train_time = time.perf_counter() - t0_train

print(f"Training completed in {lora_train_time:.2f} seconds.")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Found 120 attention projection layers to wrap with LoRA.

LoRA Model Parameter Statistics:
{
  "Total Parameters": 135436608,
  "Trainable Parameters": 921600,
  "Percentage Trainable": "0.6805%"
}



Starting LoRA Fine-Tuning...


  0%|          | 0/1250 [00:00<?, ?it/s]

{'loss': '1.466', 'grad_norm': '0.8647', 'learning_rate': '0.0003004', 'epoch': '0.4'}
{'loss': '1.301', 'grad_norm': '0.8448', 'learning_rate': '0.0001004', 'epoch': '0.8'}


  0%|          | 0/100 [00:00<?, ?it/s]

{'eval_loss': '1.309', 'eval_rougeL': '0.6419', 'eval_runtime': '11.07', 'eval_samples_per_second': '36.14', 'eval_steps_per_second': '9.035', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '145.9', 'train_samples_per_second': '34.28', 'train_steps_per_second': '8.569', 'train_loss': '1.366', 'epoch': '1'}
Training completed in 146.09 seconds.


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

In [32]:
def generate_response(model, tokenizer, user_query, max_new_tokens=128):
    # 1. Apply the Task 1.2 Prompt Format
    # We simulate a simple User -> Assistant turn
    full_prompt = f"<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n"
    
    # 2. Tokenize and generate
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.vocab['<|im_end|>'] # Stop generating at the end tag
        )
    
    # 3. Decode only the new tokens (the response)
    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    
    return response.strip()

In [33]:
test_queries = [
    "What are three healthy snacks I can have at work?",
    "Write a short, funny poem about a cat who thinks it's a dog.",
    "How do I boil an egg?"
]

models = {
    "Pretrained Base": pretrained_model,
    "Full Fine-Tuned": base_model, 
    "LoRA Fine-Tuned": lora_model
}

for query in test_queries:
    print(f"\n{'='*50}\nUSER: {query}\n{'='*50}")
    for name, model in models.items():
        res = generate_response(model, tokenizer, query)
        print(f"[{name}]:\n{res}\n{'-'*30}")


USER: What are three healthy snacks I can have at work?
[Pretrained Base]:
What are three healthy snacks I can have at work?flavorless
Folks are usually more willing to do what they want.
What are three healthy snacks I can have at work?flavorless
What are three healthy snacks I can have at work?flavorless
What are three healthy snacks I can have at work?flavorless
What are three healthy snacks I can have at work?flavorless
What are three healthy snacks I can have at work?flavorless
What are three healthy snacks I can have at work?flavorless
What are three healthy snacks I can have at work
------------------------------
[Full Fine-Tuned]:
There are three healthy snacks you can have at work: a piece of fruit, a piece of fruit, and a piece of fruit.
------------------------------
[LoRA Fine-Tuned]:
1. Bananas - great source of potassium and fiber.
2. Whole grain crackers - great source of protein and fiber.
3. Hummus - delicious dip for dipping. MetaInfoClassPlaneProtectionPlaneProtecti